# Individual Contribution – Member 1

This notebook uses the cleaned datasets exported during the data preparation stage. Place `Housing_Cleaned.csv` and `Transportation_Cleaned.csv` in the same directory as this notebook.

In [3]:
import pandas as pd
import plotly.express as px

housing_clean = pd.read_csv('datasets/Housing_Cleaned.csv')
transport_clean = pd.read_csv('datasets/Transportation_Cleaned.csv')

In [4]:
for df, date_col, name in [
    (housing_clean, 'project_completion_date', 'HOUSING'),
    (transport_clean, 'crash_date', 'TRANSPORTATION')
]:
    dates = pd.to_datetime(df[date_col], errors='coerce')
    print(f"\n{name}")
    print("Earliest:", dates.min())
    print("Latest:", dates.max())
    print("Invalid dates:", dates.isna().sum())

print("\nHOUSING BOROUGHS:")
print(housing_clean['borough'].value_counts())

print("\nTRANSPORTATION BOROUGHS:")
print(transport_clean['crash_borough'].value_counts())

print("\nHOUSING UNITS:")
print(housing_clean['all_counted_units'].describe())

print("\nSEVERE INJURIES:")
print(transport_clean['total_severe_injuries'].describe())

print("\nFATALITIES:")
print(transport_clean['total_fatalities'].describe())


HOUSING
Earliest: 2014-01-03 00:00:00
Latest: 2026-08-11 00:00:00
Invalid dates: 1819

TRANSPORTATION
Earliest: 2022-01-01 00:00:00
Latest: 2026-06-30 00:00:00
Invalid dates: 444

HOUSING BOROUGHS:
borough
BROOKLYN         3649
BRONX            2630
MANHATTAN        1485
QUEENS           1342
STATEN ISLAND     445
Name: count, dtype: int64

TRANSPORTATION BOROUGHS:
crash_borough
BROOKLYN         3954
QUEENS           3282
MANHATTAN        2733
BRONX            2617
STATEN ISLAND     669
Name: count, dtype: int64

HOUSING UNITS:
count    9522.000000
mean       33.705734
std        65.834105
min         1.000000
25%         2.000000
50%         8.000000
75%        30.000000
max       917.000000
Name: all_counted_units, dtype: float64

SEVERE INJURIES:
count    13310.000000
mean         1.011495
std          0.451514
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          7.000000
Name: total_severe_injuries, dtype: float64

FATALITIES:
count 

In [5]:
# Housing: January 2022 - June 2026
housing_period = housing_clean.copy()

housing_period['date'] = pd.to_datetime(
    housing_period['project_completion_date'],
    errors='coerce'
)

housing_period = housing_period[
    (housing_period['date'] >= '2022-01-01') &
    (housing_period['date'] < '2026-07-01')
]

housing_borough = (
    housing_period.groupby('borough', as_index=False)
    .agg(affordable_units=('all_counted_units', 'sum'))
)

# Transportation: Same period
transport_period = transport_clean.copy()

transport_period['date'] = pd.to_datetime(
    transport_period['crash_date'],
    errors='coerce'
)

transport_period = transport_period[
    (transport_period['date'] >= '2022-01-01') &
    (transport_period['date'] < '2026-07-01')
]

transport_borough = (
    transport_period.groupby('crash_borough', as_index=False)
    .agg(
        severe_injuries=('total_severe_injuries', 'sum'),
        fatalities=('total_fatalities', 'sum')
    )
)

# Secure NaN addition
transport_borough['serious_traffic_harm'] = (
    transport_borough['severe_injuries'].fillna(0) +
    transport_borough['fatalities'].fillna(0)
)

# Integrate
borough_analysis = housing_borough.merge(
    transport_borough,
    left_on='borough',
    right_on='crash_borough',
    how='inner'
)

display(borough_analysis)

# Updated visualization
import plotly.express as px
fig = px.scatter(
    borough_analysis,
    x='affordable_units',
    y='serious_traffic_harm',
    color='borough',
    text='borough',
    title='Affordable Housing vs Serious Traffic Harm (2022–2026)',
    labels={
        'affordable_units': 'Affordable Housing Units',
        'serious_traffic_harm': 'Severe Injuries + Fatalities'
    }
)

fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white')
fig.show()

,borough,affordable_units,crash_borough,severe_injuries,fatalities,serious_traffic_harm
0,BRONX,49257.0,BRONX,2659.0,209.0,2868.0
1,BROOKLYN,25839.0,BROOKLYN,3970.0,315.0,4285.0
2,MANHATTAN,19723.0,MANHATTAN,2718.0,186.0,2904.0
3,QUEENS,19493.0,QUEENS,3356.0,322.0,3678.0
4,STATEN ISLAND,640.0,STATEN ISLAND,690.0,64.0,754.0


In [6]:
# Prepare housing monthly data
housing_monthly = housing_clean.copy()

housing_monthly['month'] = pd.to_datetime(
    housing_monthly['project_completion_date'],
    errors='coerce'
).dt.to_period('M')

housing_monthly = housing_monthly.dropna(
    subset=['month', 'all_counted_units']
)

housing_monthly = housing_monthly[
    (housing_monthly['month'] >= '2022-01') &
    (housing_monthly['month'] <= '2026-06')
]

housing_monthly = (
    housing_monthly.groupby('month')['all_counted_units']
    .sum()
    .reset_index()
)

housing_monthly['month'] = housing_monthly['month'].astype(str)


# Prepare transportation monthly data
transport_monthly = transport_clean.copy()

transport_monthly['month'] = pd.to_datetime(
    transport_monthly['crash_date'],
    errors='coerce'
).dt.to_period('M')

transport_monthly = transport_monthly.dropna(
    subset=['month']
)

transport_monthly = transport_monthly[
    (transport_monthly['month'] >= '2022-01') &
    (transport_monthly['month'] <= '2026-06')
]

transport_monthly['serious_harm'] = (
    transport_monthly['total_severe_injuries'].fillna(0) +
    transport_monthly['total_fatalities'].fillna(0)
)

transport_monthly = (
    transport_monthly.groupby('month')['serious_harm']
    .sum()
    .reset_index()
)

transport_monthly['month'] = transport_monthly['month'].astype(str)


# Visualize both trends
fig = px.line(
    housing_monthly,
    x='month',
    y='all_counted_units',
    title='Monthly Affordable Housing Production in NYC',
    markers=True,
    labels={
        'month': 'Month',
        'all_counted_units': 'Affordable Housing Units'
    }
)

fig.update_layout(template='plotly_white')
fig.show()


fig = px.line(
    transport_monthly,
    x='month',
    y='serious_harm',
    title='Monthly Serious Traffic Injuries and Fatalities in NYC',
    markers=True,
    labels={
        'month': 'Month',
        'serious_harm': 'Severe Injuries + Fatalities'
    }
)

fig.update_layout(template='plotly_white')
fig.show()

# Individual Contribution – Member 1

## Research Question 1


Visualization: Scatter plot comparing affordable housing
units with serious traffic harm by borough.

Integration: Borough-level aggregation.

## Research Question 2
How have affordable housing production and serious traffic injuries changed over time?

Visualization: Two monthly trend charts.

Integration: Monthly aggregation from January 2022
to June 2026.

## Individual Contributions
- Dataset acquisition and loading.
- Exploratory data analysis.
- Data quality assessment.
- Data cleaning and standardization.
- Preparation of cleaned datasets.
- Initial borough-level integration.
- Monthly aggregation.
- Two research questions and their visualizations.